In [ ]:
%pip install numpy matplotlib

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random
%matplotlib inline

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = backward

        return out
    def __radd__(self, other): #other + self
        return self + other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other): #self - other
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward

        return out

    def __rmul__(self, other): #other * self this is for the case like 2 * a
        return self * other
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')

        def backward():
            self.grad += out.grad * other * self.data ** (other - 1)

        out._backward = backward
        return out

    def __truediv__(self, other): #self / other
        return self * other**-1

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = backward

        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def backward():
            self.grad += out.grad * out.data

        out._backward = backward
        return out
    
    def backward(self):
        #topological sort
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        # print(topo)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
            
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a*b; e.label = 'e'
d = e + c; d.label = 'd'

f = Value(-2.0, label='f')
L = d*f; L.label = 'L'
L

In [ ]:
a = Value(2.0)
b = Value(4.0)
a - b

In [ ]:
d._prev

In [ ]:
d._op

In [ ]:
import os
import sys

graphviz_path = r'C:\Program Files\Graphviz\bin'

if graphviz_path not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + graphviz_path

In [ ]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) #, node_attr={'rankdir': 'TB'})
    nodes, edges = trace(root)
    
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad ), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [ ]:
draw_dot(L)

In [ ]:
L.grad = 1.0

In [ ]:
f.grad = 4.0
d.grad = -2.0

In [ ]:
# dL / dc = dL / dd * dd / dc = -2.0 * 1.0 = -2.0
c.grad = -2.0
e.grad = -2.0

In [ ]:
# dL / da = dL / de * de / da = -2.0 * -3.0
# dL / db = dL / de * de / db = -2.0 * 2.0
a.grad = 6.0
b.grad = -4.0

In [ ]:
plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2))); plt.grid()

In [ ]:
#inputs x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

#weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias of the neuron
b = Value(6.8813735870195432, label='b')

# x1*w1 + x2*w2 + b
x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'

o = n.tanh(); o.label = 'o'

# e = (2*n).exp(); e.label = 'e'
# o = (e - 1) / (e + 1);o.label = 'o'
# o.backward()


In [ ]:
draw_dot(o)

In [ ]:
#moving the whole code in the cell below to the value class and creating a backward method to do the backpropagation
o.backward()

In [ ]:
o.grad = 1.0

#topological sort
topo = []
visited = set()

def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(o)
print(topo)

for node in reversed(topo):
    node._backward()

In [ ]:
o.grad = 1.0

In [ ]:
o._backward()

In [ ]:
n._backward()

In [ ]:
b._backward()

In [ ]:
x1w1x2w2._backward()

In [ ]:
x1w1._backward()
x2w2._backward()

In [ ]:
#manually setting the gradients for the output node

w1.grad = x1.data * x1w1.grad
x1.grad = w1.data * x1w1.grad

w2.grad = x2.data * x2w2.grad
x2.grad = w2.data * x2w2.grad

#since there is a + operator between n and b, x1w1x2w2 they both also have the same gradient as n
x1w1x2w2.grad = 0.5
b.grad = 0.5

#similarly
x1w1.grad = 0.5
x2w2.grad = 0.5

# o = tanh(n)
# do / dn = 1 - tanh^2*(n) = 1 - o^2
# do / dn = 1 - 0.707^2 = 0.5
n.grad = 0.5

o.grad = 1.0

In [ ]:
# bug in backpropagation
a = Value(3.0, label='a')
b = a + a
b.backward()
draw_dot(b)

In [ ]:
#another example
# the issue is using a variable more than once, due to overriding of the gradients. 
# so instead of setting the gradients, we need to add the gradients
a = Value(-2.0, label='a')
b = Value(3.0, label='b')

d = a * b; d.label = 'd'
e = a + b; e.label = 'e'
f = d * e; f.label = 'f'

f.backward()
draw_dot(f)

In [ ]:
%pip install torch

In [ ]:
import torch

In [ ]:
#same implementation using pytorch

x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()
print("-----")
print('x1:', x1.grad.item())
print('w1:', w1.grad.item())
print('x2:', x2.grad.item())
print('w2:', w2.grad.item())

In [ ]:
class Neuron: 
    def __init__(self, nin): 
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        #w * x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]
    

class Layer:
    #nin is the number of inputs to the layer, nout is the number of neurons in the layer
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        params = []
        for neuron in self.neurons:
            ps = neuron.parameters()
            params.extend(ps)

        return params
    
class MLP:
    #nouts is a list of the number of neurons in each layer, for example [4,4,1] means 3 layers with 4 neurons in the first layer, 4 neurons in the second layer and 1 neuron in the third layer
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [ ]:
#x = [2.0, 3.0, -1.0]
n = MLP(3, [4,4,1])
# n(x)

In [ ]:
len(n.parameters()) #these are the number of paramters that need to be tuned

In [ ]:
xs = [ #input values
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5], 
    [0.5, 1.0, 1.0], 
    [1.0, 1.0, -1.0], 
]

ys = [1.0, -1.0, -1.0, 1.0] #desired targets


In [ ]:
#training loop
for k in range(20):
    #forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))

    #backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    #update
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss)